<a href="https://colab.research.google.com/github/devarahaasan/Telecom-AI-Brand-Intelligence-System/blob/main/mainstrem1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**streamlit (API)**

In [ ]:
!pip install streamlit fastapi uvicorn nest-asyncio pyngrok sentence-transformers faiss-cpu joblib

In [ ]:
%%writefile AI.py
import streamlit as st
import joblib
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from fastapi import FastAPI
from pydantic import BaseModel

# 1. Initialize FastAPI for the API layer
app = FastAPI()

# 2. Define the structure for API requests (Input must be text)
class FeedbackInput(BaseModel):
    text: str

# 3. Use Streamlit's cache to load heavy models only once to save memory
@st.cache_resource
def load_models():
    # Load the Sentiment Analysis model (Pickle file)
    sentiment_pipeline = joblib.load("sentiment_pipeline.pkl")

    # Load the FAISS vector database index
    index = faiss.read_index("faiss_index.bin")

    # Load the text documents associated with the vector index
    documents = joblib.load("documents.pkl")

    # Load the Transformer model used for creating text embeddings
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

    return sentiment_pipeline, index, documents, embedding_model

# 4. Trigger the model loading process
sentiment_pipeline, index, documents, embedding_model = load_models()

# 5. Define the API Endpoint for external applications
@app.post("/predict_api")
def predict_api(data: FeedbackInput):
    # Process the text sent via API using the sentiment model
    res = sentiment_pipeline(data.text)
    # Return the sentiment result as a JSON response
    return {"sentiment": res['label'], "status": "Success"}

# 6. Streamlit User Interface (UI) starts here
st.title("📡 Telecom AI Dashboard")

# 7. Create a text area for users to type customer feedback
user_input = st.text_area("Enter Feedback")

# 8. Create a button to trigger the analysis on the dashboard
if st.button("Analyze"):
    # Check if input is not empty
    if user_input.strip() != "":
        st.success("Analysis Complete!")
    else:
        st.warning("Please enter some text first.")

In [ ]:
import gradio as gr
import joblib
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# --- STEP 1: LOAD ALL MODELS ---
sentiment_model = joblib.load("sentiment_pipeline.pkl")
index = faiss.read_index("faiss_index.bin")
documents = joblib.load("documents.pkl")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Ensure documents is a clean list
if not isinstance(documents, list):
    documents = list(documents)

# --- STEP 2: SUPER DARK THEME CSS (Force All White Text) ---
custom_css = """
/* Background & Main Container */
.gradio-container {
    background-color: #0E1117 !important;
}

/* Force All Labels, Headings, and Paragraphs to be Pure White */
label, p, span, h1, h2, h3, .label-content, .secondary-label {
    color: #FFFFFF !important;
    font-weight: 600 !important;
}

/* Input & Output Boxes Styling */
textarea, input {
    background-color: #1E1E1E !important;
    color: #FFFFFF !important;
    border: 2px solid #00ADB5 !important;
    font-size: 16px !important;
}

/* Fix for predicted text color inside output boxes */
.border-none {
    color: #FFFFFF !important;
}

/* Primary Button Styling */
button.primary {
    background: linear-gradient(90deg, #00ADB5, #007B83) !important;
    color: white !important;
    border: none !important;
    font-weight: bold !important;
    font-size: 18px !important;
    padding: 10px !important;
}

/* Ensure placeholder text is visible but slightly grey */
::placeholder {
    color: #888888 !important;
}
"""

# --- STEP 3: CORE LOGIC ---
def analyze_feedback(user_input):
    try:
        if not user_input.strip():
            return "⚠️ Empty Input", "Please provide customer feedback to analyze."

        # A. Sentiment Analysis
        res = sentiment_model(user_input)
        # Handle different pipeline output formats
        sentiment = res[0]['label'] if isinstance(res, list) else res['label']

        # B. RAG Process (FAISS Search)
        query_vector = embedder.encode([user_input]).astype('float32')
        distances, indices = index.search(query_vector, k=1)
        best_index = int(indices[0][0])

        # Get matching solution
        if best_index != -1 and best_index < len(documents):
            suggested_solution = documents[best_index]
        else:
            suggested_solution = "No matching telecom protocol found for this issue."

        return sentiment, suggested_solution

    except Exception as e:
        return f"Error: {str(e)}", "Please check model files."

# --- STEP 4: UI DESIGN (STYLISH LAYOUT) ---
with gr.Blocks(css=custom_css, title="Telecom AI Brand Intel") as demo:
    gr.Markdown("<h1 style='text-align: center;'>📡 Telecom AI Brand Intelligence System</h1>")
    gr.Markdown("<p style='text-align: center; font-size: 1.2em;'>Enterprise-Grade Sentiment Analysis & RAG Solution</p>")

    with gr.Row():
        with gr.Column(scale=2):
            input_text = gr.Textbox(
                label="💬 Enter Customer Feedback",
                placeholder="Example: My 5G speed is very slow and I am being charged for roaming...",
                lines=5
            )
            submit_btn = gr.Button("ANALYZE FEEDBACK 🚀", variant="primary")

        with gr.Column(scale=2):
            output_sentiment = gr.Textbox(label="📊 Predicted Sentiment Status", interactive=False)
            output_solution = gr.Textbox(label="💡 AI Suggested Solution (RAG)", lines=5, interactive=False)

    gr.Markdown("---")
    gr.Markdown("<p style='text-align: center; color: #888 !important;'>Powered by HuggingFace Transformers | FAISS Vector DB | Python</p>")

    # Binding the button
    submit_btn.click(
        fn=analyze_feedback,
        inputs=input_text,
        outputs=[output_sentiment, output_solution]
    )

# --- STEP 5: DEPLOY ---
demo.launch(share=True)

In [ ]:
from gradio_client import Client

# 1. Establish a connection to your live AI model hosted on Gradio
# Replace the URL with your current active live link
client = Client("https://2f42a7dfaf967a3fca.gradio.live/")

# 2. Send a request to the API with customer feedback
# The 'feedback' parameter is sent to the /predict endpoint
result = client.predict(
    feedback="The network coverage is very poor in my area",
    api_name="/predict"
)

# 3. Print the AI's analysis result to the console
print(result)

**output examples**

**1 Star: Extremely Angry / Frustrated**


"I am absolutely disgusted with this service! My phone has had no signal for three days and your support team is useless. I want to cancel my subscription immediately and get a full refund. This is the worst company ever!"

**Keywords**: disgusted, useless, cancel, worst.


**2 Stars: Dissatisfied / Unhappy**


"The internet speed is very inconsistent lately. It keeps dropping during my office meetings, and the monthly bill is much higher than what was promised. I am not happy with how things are going."


**Keywords**: inconsistent, dropping, higher bill, not happy.


**3 Stars: Neutral / Indifferent**


"The service is okay, I guess. It works most of the time, but it's nothing special. I haven't had any major problems, but I haven't been impressed either. It’s just an average network."


**Keywords**: okay, most of the time, nothing special, average.


**4 Stars: Satisfied / Good**


"The 5G coverage in my area is quite good and I rarely face any call drops. The mobile app is easy to use for paying bills. Overall, I am satisfied with the network performance."


**Keywords**: quite good, rarely, easy to use, satisfied.


**5 Stars: Excellent / Very Happy**


"Wow, I am amazed by the lightning-fast 5G speeds! The customer service agent was incredibly helpful and resolved my query in minutes. I would highly recommend this provider to everyone. Truly excellent service!"


**Keywords**: amazed, lightning-fast, incredibly helpful, highly recommend.
